# Per-Bone Multi-Label Modelling (4 bones) — learnability sanity-check

Adapts the binary-occupancy decoder pipeline (`decoder_pipeline.ipynb`) to a 4-channel
multi-label task: femur / tibia / patella / fibula, one channel per bone, on the
fractured (Ruikar) per-bone ground truth only.

Cells 2–6 are copied **verbatim** from `decoder_pipeline.ipynb` (shared config + encoder/fusion
front-end). The model code (`SuperResHead`, `Decoder3D`) is then generalised from 1 output
channel to `N_CLASSES=4`. A new per-bone GT loader and a restricted paired index follow. This
notebook ends at a shape-validation check — training/loss is Task 6.

In [1]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # reduce CUDA fragmentation (OOM safeguard); must be set before torch initialises CUDA
import math, random, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as cp
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
import nibabel as nib
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("torch", torch.__version__, "| timm", timm.__version__, "| cuda:", torch.cuda.is_available())

torch 2.10.0+cpu | timm 1.0.27 | cuda: False


In [2]:
# ============================= CONFIG (LOCAL - smoke test) =============================
# The ONLY cell you normally edit. Keep this fast: it just proves the pipeline runs.
ENV          = "local"
DEVICE       = torch.device("cpu")
MODEL        = "unet"        # <- switch to "vnet" and re-run to train the comparison decoder
TARGET_RES   = 64            # smoke-test volume size (HPC twin uses 256)
LIFT_DEPTH   = 16            # encoder 2D->3D lift depth (matches encoder_pipeline.ipynb)
EPOCHS       = 2
BATCH_SIZE   = 1
LR           = 1e-4
CKPT_EVERY   = 1             # save a revisitable checkpoint every N epochs
USE_AMP      = False         # mixed precision: CUDA only
USE_GRAD_CKPT = False        # gradient checkpointing: only needed for large volumes
NUM_WORKERS  = 0             # 0 is safest on Windows
GT_THRESH    = 0.40          # bone threshold on the [0,1] windowed CT (tune in the QA cell)
INCLUDE_GEOMETRIC = False    # exclude rotated/flipped DRRs (their 3D GT would need the same transform)
DEEP_SUPERVISION  = False    # optional aux losses at coarse decoder scales
# --- cross-validation (knee-level, dataset-stratified) ---
N_FOLDS      = 5             # k-fold CV over knees (matches FracReconNet); every knee is tested once
FOLD         = 0             # which fold is held out as TEST this run (0..N_FOLDS-1); smoke: keep 0
# --- front-end regime (run BOTH and compare in decoder_comparison.ipynb) ---
REGIME       = "frozen"      # "frozen": strict decoder isolation (front-end frozen -> byte-identical
                             #   features for U-Net & V-Net). "finetuned": train encoder+fusion+lift
                             #   +decoder jointly (realistic capacity; NOT a pure decoder ablation).
FREEZE_FRONTEND = (REGIME == "frozen")   # derived from REGIME so the two regimes stay consistent
FREEZE_ENCODER  = (REGIME == "frozen")   # finetuned regime trains the SimCLR backbone too
PRETRAINED   = True          # ImageNet init for the backbone (falls back to random if offline)
SMOKE_TEST   = True          # subsample to a few cases for a quick check (use FOLD=0 in smoke)
SMOKE_CASES_PER_GROUP = 3
RESUME_FROM  = None          # path to a checkpoint .pth to resume from, else None
EXPLICIT_ROOT = None         # set a path string only if root auto-detection fails
print("ENV", ENV, "| MODEL", MODEL, "| fold", FOLD, "/", N_FOLDS, "| regime", REGIME,
      "| TARGET_RES", TARGET_RES, "| device", DEVICE, "| epochs", EPOCHS)

ENV local | MODEL unet | fold 0 / 5 | regime frozen | TARGET_RES 64 | device cpu | epochs 2


In [3]:
# Resolve the project root robustly (works locally and on HPC, regardless of where
# the notebook is launched from). We look upward for the data/interim/predrr folder,
# which holds the ground-truth CT volumes.
def find_root(start: Path) -> Path:
    if EXPLICIT_ROOT:
        r = Path(EXPLICIT_ROOT)
        if (r / "data" / "interim" / "predrr").exists():
            return r
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "data" / "interim" / "predrr").exists():
            return cand
    raise FileNotFoundError("Could not find project root (expected data/interim/predrr). "
                            "Set EXPLICIT_ROOT in the CONFIG cell.")

ROOT           = find_root(Path.cwd())
DATA           = ROOT / "data"
NORMAL_DRR_DIR = DATA / "interim" / "DRRs"                 # AP/LAT DRRs (model inputs)
AUG_DRR_DIR    = DATA / "processed" / "augmented_DRRs"     # augmented DRR variants
PREDRR_DIR     = DATA / "interim" / "predrr"               # ground-truth CT volumes
MODELS_DIR     = ROOT / "models"
SIMCLR_CKPT    = MODELS_DIR / "convnextv2_simclr_encoder.pth"      # SimCLR encoder (fold-agnostic; SSL uses no labels)
FRONTEND_CKPT  = MODELS_DIR / ("front_end_fold%d.pth" % FOLD)      # per-fold frozen front-end (frozen regime only)
# per-fold + per-regime + per-model checkpoints, so the 2 models x 2 regimes per fold never collide
CKPT_DIR       = MODELS_DIR / "decoders" / ("fold%d" % FOLD) / REGIME / MODEL
CKPT_DIR.mkdir(parents=True, exist_ok=True)
GT_CACHE_DIR   = DATA / "interim" / ("predrr_occupancy_%d" % TARGET_RES)   # cached binary GT
print("ROOT:", ROOT)
print("checkpoints ->", CKPT_DIR)

ROOT: C:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject
checkpoints -> C:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject\models\decoders\fold0\frozen\unet


## 1. Shared encoder (copied verbatim from `decoder_pipeline.ipynb`)

In [4]:
# ===== Encoder front-end - VERBATIM from decoder_pipeline.ipynb. DO NOT EDIT. =====
# (PRETRAINED / FREEZE_ENCODER are set in the CONFIG cell so they stay visible knobs.)
BACKBONE     = "convnextv2_tiny"
IMG_SIZE     = 256
OUT_CHANNELS = [64, 128, 256, 512]
FUSION_TYPES = ["local", "local", "attn", "attn"]   # fine -> coarse

def make_backbone(pretrained=True):
    """features_only ConvNeXtV2 returning 4 multi-scale maps. Falls back to random init offline."""
    try:
        return timm.create_model(BACKBONE, pretrained=pretrained, features_only=True)
    except Exception as e:
        print("[warn] pretrained fetch failed (%s); random init." % type(e).__name__)
        return timm.create_model(BACKBONE, pretrained=False, features_only=True)

FEAT_DIMS = [f["num_chs"] for f in make_backbone(pretrained=False).feature_info]   # [96,192,384,768]

def load_drr(path):
    """npy 256x256 float32 [0,1] -> tensor [3,H,W] (1 channel replicated to 3 for ConvNeXtV2)."""
    arr = np.load(path).astype(np.float32)
    t = torch.from_numpy(arr)
    if t.ndim == 2:
        t = t.unsqueeze(0)
    return t.repeat(3, 1, 1) if t.shape[0] == 1 else t

NORMALIZE = T.Normalize(mean=[0.5] * 3, std=[0.5] * 3)
def paired_tf(t):
    return NORMALIZE(t)

class CrossAttention(nn.Module):
    """AP (query) attends to LAT (key/value). Operates on tokens [B, N, C]."""
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim); self.k = nn.Linear(dim, dim); self.v = nn.Linear(dim, dim)
        self.scale = dim ** -0.5
    def forward(self, a, b):
        attn = F.softmax(torch.matmul(self.q(a), self.k(b).transpose(-2, -1)) * self.scale, dim=-1)
        return torch.matmul(attn, self.v(b)) + a

class LocalFusion(nn.Module):
    """Cheap high-res fusion: concat views + 3x3 conv, residual on AP."""
    def __init__(self, dim):
        super().__init__()
        self.mix = nn.Conv2d(2 * dim, dim, kernel_size=3, padding=1)
    def forward(self, a, b):
        return self.mix(torch.cat([a, b], dim=1)) + a

# --- bi-planar lift orientation (resolved empirically; see Check 1 / _axis_probe) ---
# GT array axes, from nibabel axcodes ('L','P','S'): axis0 = L-R, axis1 = A-P, axis2 = S-I.
# AP projects along A-P (axis1); LAT projects along L-R (axis0). Both DRR rows (H) = S-I (axis2);
# AP cols (W) = L-R (axis0); LAT cols (W) = A-P (axis1). The fused cube is ordered (axis0, axis1,
# axis2) to match the GT array. flip_* reverse a row/col direction vs its volume axis; locked from
# the affine + 1-D S-I profile test and re-confirmed by the one-sample overfit guard.
LIFT_FLIP_SI      = False   # DRR rows  vs axis2 (S-I)
LIFT_FLIP_AP_COL  = False   # AP  cols  vs axis0 (L-R)
LIFT_FLIP_LAT_COL = True    # LAT cols  vs axis1 (A-P)

class BiPlanarFeatureFusion(nn.Module):
    def __init__(self, feat_dims=FEAT_DIMS, out_channels=OUT_CHANNELS,
                 fusion_types=FUSION_TYPES, depth=16, pretrained=True, freeze_encoder=False):
        super().__init__()
        self.encoder = make_backbone(pretrained)
        self.fusion_types = list(fusion_types)
        self.depth = depth   # retained for signature compat; the orthogonal lift no longer uses it
        self.fuse = nn.ModuleList([CrossAttention(d) if t == "attn" else LocalFusion(d)
                                   for d, t in zip(feat_dims, fusion_types)])
        self.to3d = nn.ModuleList([nn.Conv2d(c, o, 1) for c, o in zip(feat_dims, out_channels)])
        # expand3d fuses the two orthogonally back-projected view cubes (2*o -> o) in 3D
        self.expand3d = nn.ModuleList([nn.Conv3d(2 * o, o, 3, padding=1) for o in out_channels])
        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False
    def load_simclr_encoder(self, path):
        missing, unexpected = self.encoder.load_state_dict(torch.load(path, map_location="cpu"), strict=False)
        print("loaded SimCLR encoder: missing=%d unexpected=%d" % (len(missing), len(unexpected)))
    def _ortho_lift(self, ap_f, lat_f, c2d, c3d):
        """Orthogonal back-projection lift. Each view is placed on the two volume axes it resolves
        and broadcast along its (unobserved) projection axis; the two view cubes are then fused in
        3D. Preserves bi-planar depth instead of extruding one fused 2D map along Z. Output cube is
        ordered (axis0=L-R, axis1=A-P, axis2=S-I) to match the GT array."""
        B, C, H, W = ap_f.shape           # square feature map at this level: S = H = W
        S = H
        ap = c2d(ap_f); lat = c2d(lat_f)  # shared 1x1 projection -> [B, O, H, W] each
        O = ap.shape[1]
        if LIFT_FLIP_SI:      ap = ap.flip(2); lat = lat.flip(2)   # rows (H) = S-I (axis2)
        if LIFT_FLIP_AP_COL:  ap = ap.flip(3)                       # AP  cols (W) = L-R (axis0)
        if LIFT_FLIP_LAT_COL: lat = lat.flip(3)                     # LAT cols (W) = A-P (axis1)
        # AP: (H=axis2, W=axis0) -> cube (axis0, axis1, axis2), broadcast over axis1 (A-P)
        ap_cube = ap.permute(0, 1, 3, 2).unsqueeze(3).expand(B, O, S, S, S)
        # LAT: (H=axis2, W=axis1) -> cube (axis0, axis1, axis2), broadcast over axis0 (L-R)
        lat_cube = lat.permute(0, 1, 3, 2).unsqueeze(2).expand(B, O, S, S, S)
        return c3d(torch.cat([ap_cube, lat_cube], dim=1))           # [B, O, S, S, S]
    def forward(self, ap_img, lat_img):
        ap_feats, lat_feats = self.encoder(ap_img), self.encoder(lat_img)
        fused2d, fused3d = [], []
        for ap_f, lat_f, fuse, c2d, c3d, t in zip(
                ap_feats, lat_feats, self.fuse, self.to3d, self.expand3d, self.fusion_types):
            B, C, H, W = ap_f.shape
            if t == "attn":
                a = ap_f.flatten(2).transpose(1, 2); b = lat_f.flatten(2).transpose(1, 2)
                f2d = fuse(a, b).transpose(1, 2).reshape(B, C, H, W)
            else:
                f2d = fuse(ap_f, lat_f)
            fused2d.append(f2d)                                  # kept only for feature-viz cells
            fused3d.append(self._ortho_lift(ap_f, lat_f, c2d, c3d))
        return fused2d, fused3d

print("encoder feature dims:", FEAT_DIMS)

encoder feature dims: [96, 192, 384, 768]


## 2. Multi-label config + generalised decoder (4-bone head)

In [5]:
# Multi-label extension: 4 output channels, one per bone.
# N_CLASSES must be defined BEFORE the model-class cell runs.
N_CLASSES = 4
BONES     = ["femur", "tibia", "patella", "fibula"]
print("N_CLASSES:", N_CLASSES, "| BONES:", BONES)

N_CLASSES: 4 | BONES: ['femur', 'tibia', 'patella', 'fibula']


In [6]:
# ===== Decoder blocks — generalised for N_CLASSES output channels =====
# Copied from decoder_pipeline.ipynb cell 11; only the Conv3d output channels changed:
#   SuperResHead.out : nn.Conv3d(8, 1, 1) -> nn.Conv3d(8, N_CLASSES, 1)
#   Decoder3D.aux*   : nn.Conv3d(cX, 1, 1) -> nn.Conv3d(cX, N_CLASSES, 1)
# Everything else (block wiring, skip connections, SuperRes staging) is unchanged.

def conv_block(block_type, in_ch, out_ch):
    return DoubleConv(in_ch, out_ch) if block_type == "unet" else VNetResBlock(in_ch, out_ch)

class DoubleConv(nn.Module):
    """U-Net block: (Conv3d -> BN -> ReLU) x2. Plain, no residual."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.net(x)

class VNetResBlock(nn.Module):
    """V-Net block: (Conv3d -> BN -> PReLU) x2 + residual add (input projected if channels differ)."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.proj = nn.Conv3d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.c1 = nn.Conv3d(in_ch, out_ch, 3, padding=1); self.n1 = nn.BatchNorm3d(out_ch); self.a1 = nn.PReLU(out_ch)
        self.c2 = nn.Conv3d(out_ch, out_ch, 3, padding=1); self.n2 = nn.BatchNorm3d(out_ch); self.a2 = nn.PReLU(out_ch)
    def forward(self, x):
        y = self.a1(self.n1(self.c1(x)))
        y = self.n2(self.c2(y))
        return self.a2(y + self.proj(x))

class SuperResHead(nn.Module):
    """Grow the (S, S, S) cube feature grid up to (T,T,T) via staged trilinear upsample + refine
    blocks with tapering channels (heavy work stays at low resolution -> low memory).
    Output: (B, N_CLASSES, T, T, T) — one logit map per bone."""
    def __init__(self, block_type, in_ch, target, use_grad_ckpt=False):
        super().__init__()
        self.target = tuple(int(t) for t in target); self.use_grad_ckpt = use_grad_ckpt
        self.b1 = conv_block(block_type, in_ch, 32)
        self.b2 = conv_block(block_type, 32, 16)
        self.b3 = conv_block(block_type, 16, 8)
        self.out = nn.Conv3d(8, N_CLASSES, 1)   # <-- generalised: was nn.Conv3d(8, 1, 1)
    def _run(self, blk, x):
        if self.use_grad_ckpt and x.requires_grad:
            return cp.checkpoint(blk, x, use_reentrant=False)
        return blk(x)
    def forward(self, x):
        d0, h0, w0 = x.shape[-3:]; dt, ht, wt = self.target
        s1 = (round(d0 + (dt - d0) / 3), round(h0 + (ht - h0) / 3), round(w0 + (wt - w0) / 3))
        s2 = (round(d0 + 2 * (dt - d0) / 3), round(h0 + 2 * (ht - h0) / 3), round(w0 + 2 * (wt - w0) / 3))
        x = F.interpolate(x, size=s1, mode="trilinear", align_corners=False); x = self._run(self.b1, x)
        x = F.interpolate(x, size=s2, mode="trilinear", align_corners=False); x = self._run(self.b2, x)
        x = F.interpolate(x, size=self.target, mode="trilinear", align_corners=False); x = self._run(self.b3, x)
        return self.out(x)

class Decoder3D(nn.Module):
    """Multi-scale skip-connected decoder. Same wiring for both models; only the block differs.
    Aux heads (deep supervision) output N_CLASSES channels to match the main head.
    Output: (B, N_CLASSES, T, T, T)."""
    def __init__(self, block_type, enc_channels=OUT_CHANNELS, target=(64, 64, 64),
                 use_grad_ckpt=False, deep_supervision=False):
        super().__init__()
        c0, c1, c2, c3 = enc_channels
        self.deep_supervision = deep_supervision; self.target = tuple(int(t) for t in target)
        self.up3 = nn.ConvTranspose3d(c3, c2, kernel_size=2, stride=2)   # 8^3 -> 16^3 (symmetric)
        self.dec3 = conv_block(block_type, c2 + c2, c2)
        self.up2 = nn.ConvTranspose3d(c2, c1, kernel_size=2, stride=2)   # 16^3 -> 32^3
        self.dec2 = conv_block(block_type, c1 + c1, c1)
        self.up1 = nn.ConvTranspose3d(c1, c0, kernel_size=2, stride=2)   # 32^3 -> 64^3
        self.dec1 = conv_block(block_type, c0 + c0, c0)
        self.sr = SuperResHead(block_type, c0, self.target, use_grad_ckpt)
        if deep_supervision:
            # <-- generalised: was nn.Conv3d(cX, 1, 1)
            self.aux3 = nn.Conv3d(c2, N_CLASSES, 1)
            self.aux2 = nn.Conv3d(c1, N_CLASSES, 1)
            self.aux1 = nn.Conv3d(c0, N_CLASSES, 1)
    def forward(self, feats):
        l0, l1, l2, l3 = feats
        x = self.up3(l3); x = torch.cat([x, l2], 1); x = self.dec3(x); a3 = x
        x = self.up2(x);  x = torch.cat([x, l1], 1); x = self.dec2(x); a2 = x
        x = self.up1(x);  x = torch.cat([x, l0], 1); x = self.dec1(x); a1 = x
        out = self.sr(x)
        if self.deep_supervision and self.training:
            up = lambda h: F.interpolate(h, size=self.target, mode="trilinear", align_corners=False)
            return out, [up(self.aux3(a3)), up(self.aux2(a2)), up(self.aux1(a1))]
        return out, None

class ReconModel(nn.Module):
    """Full model = shared bi-planar encoder/fusion + a (U-Net or V-Net) decoder."""
    def __init__(self, fusion, decoder):
        super().__init__(); self.fusion = fusion; self.decoder = decoder
    def forward(self, ap, lat):
        _, f3d = self.fusion(ap, lat)
        return self.decoder(f3d)

In [7]:
def build_model():
    # The fusion (encoder + bi-planar fusion + 2D->3D lift) is built once; how we initialise and
    # freeze it depends on the comparison mode.
    fusion = BiPlanarFeatureFusion(depth=LIFT_DEPTH, pretrained=PRETRAINED,
                                   freeze_encoder=(FREEZE_ENCODER and not FREEZE_FRONTEND))
    if FREEZE_FRONTEND and FRONTEND_CKPT.exists():
        # STRICT comparison: load the pretrained front-end and freeze it WHOLESALE, so both the
        # U-Net and V-Net runs consume byte-identical features (only the decoder differs).
        sd = torch.load(FRONTEND_CKPT, map_location="cpu")["front_end"]
        missing, unexpected = fusion.load_state_dict(sd, strict=False)
        for p in fusion.parameters():
            p.requires_grad = False
        print("loaded FROZEN front-end from %s: missing=%d unexpected=%d"
              % (FRONTEND_CKPT.name, len(missing), len(unexpected)))
    elif SIMCLR_CKPT.exists():
        # FALLBACK: only the SimCLR encoder is pretrained; fusion+lift train with the decoder.
        if FREEZE_FRONTEND:
            print("[warn] FREEZE_FRONTEND=True but %s not found; run frontend_pretrain.ipynb first. "
                  "Falling back to SimCLR-encoder-only." % FRONTEND_CKPT.name)
        fusion.load_simclr_encoder(SIMCLR_CKPT)
    else:
        print("[warn] no front-end or SimCLR checkpoint; encoder uses ImageNet/random init.")
    decoder = Decoder3D(MODEL, target=(TARGET_RES,) * 3,
                        use_grad_ckpt=USE_GRAD_CKPT, deep_supervision=DEEP_SUPERVISION)
    if USE_GRAD_CKPT and not FREEZE_FRONTEND:
        try:
            fusion.encoder.set_grad_checkpointing(True)
            print("encoder gradient checkpointing: ON (finetuned regime)")
        except Exception as e:
            print("[warn] encoder grad checkpointing unavailable (%s)." % type(e).__name__)
    return ReconModel(fusion, decoder)

## 3. Per-bone GT loader

In [8]:
import re

GT_PB = ROOT / "data" / "interim" / "gt_per_bone_256" / "fractured"
KEYS  = sorted(p.name for p in GT_PB.iterdir() if p.is_dir())   # ['Case1_PartLeft', ...]
print(len(KEYS), "per-bone GT keys:", KEYS)

def key_from(dataset, case, side):
    Side = "Right" if str(side).lower().startswith("r") else "Left"
    return f"{case}_Part{Side}"

def load_gt_per_bone(dataset, case, side):
    key = key_from(dataset, case, side)
    chans = []
    for b in BONES:
        vol = nib.load(str(GT_PB / key / f"{key}_{b}.nii.gz")).get_fdata().astype(np.float32)
        occ = (vol > 0.5).astype(np.float32)
        t = F.interpolate(torch.from_numpy(occ)[None, None], size=(TARGET_RES,) * 3, mode="nearest")
        chans.append(t[0, 0])
    return torch.stack(chans)   # (4, T, T, T)

11 per-bone GT keys: ['Case11_PartRight', 'Case12_PartRight', 'Case13_PartRight', 'Case16_PartRight', 'Case1_PartLeft', 'Case2_PartLeft', 'Case3_PartLeft', 'Case5_PartRight', 'Case6_PartRight', 'Case7_PartRight', 'Case9_PartRight']


## 4. Paired index + per-bone dataset (fractured only)

In [9]:
def build_paired_index():
    """One row per (case, side, variant) with absolute AP/LAT paths + metadata."""
    rows = []
    nmeta = pd.read_csv(NORMAL_DRR_DIR / "drr_generation_metadata.csv")
    for (ds, case, side), _ in nmeta.groupby(["dataset", "case", "side"]):
        ap = NORMAL_DRR_DIR / ds / case / side / "ap.npy"
        lat = NORMAL_DRR_DIR / ds / case / side / "lat.npy"
        if ap.exists() and lat.exists():
            rows.append(dict(dataset=ds, case=case, side=side, variant="normal",
                             geometric=False, ap=str(ap), lat=str(lat)))
    ameta_path = AUG_DRR_DIR / "augmentation_variants_metadata.csv"
    if ameta_path.exists():
        ameta = pd.read_csv(ameta_path)
        for r in ameta.itertuples(index=False):
            ap = AUG_DRR_DIR / r.ap_npy; lat = AUG_DRR_DIR / r.lat_npy
            if ap.exists() and lat.exists():
                rows.append(dict(dataset=r.dataset, case=r.case, side=r.side, variant=r.variant,
                                 geometric=bool(r.geometric), ap=str(ap), lat=str(lat)))
    return pd.DataFrame(rows)

# Build the full index, drop geometric augmentations if excluded, then restrict to
# fractured cases that have per-bone GT available.
paired_index = build_paired_index()
if not INCLUDE_GEOMETRIC:
    paired_index = paired_index[~paired_index.geometric].reset_index(drop=True)

paired_index = paired_index[paired_index.apply(
    lambda r: r.dataset == "fractured" and key_from(r.dataset, r.case, r.side) in KEYS, axis=1
)].reset_index(drop=True)
print("fractured per-bone rows:", len(paired_index),
      "| knees:", paired_index.groupby(["case", "side"]).ngroups)

class PairedDRRVolumeDataset(Dataset):
    """Returns AP/LAT DRRs (3x256x256) + per-bone GT occupancy (4,T,T,T) + metadata."""
    def __init__(self, df, transform=paired_tf):
        self.df = df.reset_index(drop=True); self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        ap  = self.transform(load_drr(r.ap))
        lat = self.transform(load_drr(r.lat))
        gt  = load_gt_per_bone(r.dataset, r.case, r.side)   # (4, T, T, T)
        return {"ap": ap, "lat": lat, "gt": gt,
                "dataset": r.dataset, "case": r.case, "side": r.side, "variant": r.variant}

fractured per-bone rows: 42 | knees: 11


## 5. Shape-validation test (Cell E)

In [10]:
m = build_model().to(DEVICE).eval()
ap = torch.randn(1, 3, 256, 256).to(DEVICE); lat = torch.randn(1, 3, 256, 256).to(DEVICE)
with torch.no_grad():
    out = m(ap, lat)
out0 = out[0] if isinstance(out, (tuple, list)) else out
assert out0.shape[1] == 4, f"expected 4 output channels, got {tuple(out0.shape)}"
assert tuple(out0.shape[2:]) == (TARGET_RES,) * 3, f"expected {TARGET_RES}^3, got {tuple(out0.shape)}"
r0 = paired_index.iloc[0]
gt = load_gt_per_bone(r0.dataset, r0.case, r0.side)
assert tuple(gt.shape) == (4,) + (TARGET_RES,) * 3, tuple(gt.shape)
print("OK | model out:", tuple(out0.shape), "| gt:", tuple(gt.shape), "| rows:", len(paired_index))

loaded FROZEN front-end from front_end_fold0.pth: missing=0 unexpected=0


OK | model out: (1, 4, 64, 64, 64) | gt: (4, 64, 64, 64) | rows: 42


## 6. Per-channel loss/metrics + learnability sanity-check (local smoke test)

Goal: prove the encoder–decoder can represent **each of the 4 bones** as an independent label.
Protocol: take 2 fractured knees and **overfit** them (train == eval == those knees); confirm
per-bone Dice climbs high for all four channels. The front-end is frozen (`REGIME="frozen"`), so the
fused 3D features are constant per input — we precompute them **once** and train only the decoder,
**batch 1** (one knee at a time) to keep peak CPU memory low.

This is the **local smoke test** (per the HPC/local split): a capacity check at `TARGET_RES=64` on
CPU. The real multi-label training — 256³, k-fold, both front-end regimes, all 11 knees — is HPC/GPU
work and is out of scope for this notebook.

In [ ]:
# --- per-channel multi-label loss + per-bone Dice metric ---
class DiceBCEMC(nn.Module):
    """Multi-channel BCE + soft-Dice, averaged over the 4 bone channels."""
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth
    def _dice(self, logits, target):
        p = torch.sigmoid(logits.float()); t = target.float()
        p = p.reshape(p.size(0), p.size(1), -1); t = t.reshape(t.size(0), t.size(1), -1)
        inter = (p * t).sum(-1)
        d = (2 * inter + self.smooth) / (p.sum(-1) + t.sum(-1) + self.smooth)
        return 1 - d.mean()
    def forward(self, logits, target):
        return 0.5 * F.binary_cross_entropy_with_logits(logits, target.float()) + 0.5 * self._dice(logits, target)

LOSS = DiceBCEMC()

def per_bone_dice(logits, target, thr=0.5):
    """Returns a length-4 array: hard Dice per bone channel, averaged over the batch."""
    p = (torch.sigmoid(logits.float()) > thr).float()
    p = p.reshape(p.size(0), p.size(1), -1); t = target.float().reshape(target.size(0), target.size(1), -1)
    inter = (p * t).sum(-1)
    d = (2 * inter + 1e-6) / (p.sum(-1) + t.sum(-1) + 1e-6)
    return d.mean(0).cpu().numpy()

print("loss + per-bone Dice metric ready")

In [ ]:
# --- overfit SMOKE test (local): 2 fractured knees, decoder-only, BATCH 1 (memory-light) ---
# Heavy training (256^3, k-fold, both regimes) is HPC work; locally we only prove learnability at
# TARGET_RES=64 with batch 1 so peak CPU memory stays low. The front-end is frozen, so we cache the
# fused 3D features once and train just the decoder.
SANITY_KEYS = KEYS[:2]
sdf = paired_index[paired_index.apply(
    lambda r: key_from(r.dataset, r.case, r.side) in SANITY_KEYS and r.variant == "normal", axis=1
)].drop_duplicates(["case", "side"]).reset_index(drop=True)
print("sanity knees:", [key_from(r.dataset, r.case, r.side) for r in sdf.itertuples()])

model = build_model().to(DEVICE); model.eval()
samples = []   # [(f3d_list (batch 1), gt (1,4,T,T,T)), ...]
with torch.no_grad():
    for r in sdf.itertuples():
        ap  = paired_tf(load_drr(r.ap))[None].to(DEVICE)
        lat = paired_tf(load_drr(r.lat))[None].to(DEVICE)
        _, f3d = model.fusion(ap, lat)
        gt = load_gt_per_bone(r.dataset, r.case, r.side)[None].to(DEVICE)
        samples.append(([f.detach() for f in f3d], gt))
print("cached", len(samples), "samples | feat0", tuple(samples[0][0][0].shape), "| gt", tuple(samples[0][1].shape))

opt = torch.optim.Adam([p for p in model.decoder.parameters() if p.requires_grad], lr=1e-3)
EPOCHS_SANITY, EVAL_EVERY, TARGET_DICE = 200, 10, 0.90
hist = []; t0 = time.time()
for ep in range(EPOCHS_SANITY):
    model.decoder.train()
    for f3d, gt in samples:                       # batch 1, one knee at a time
        opt.zero_grad()
        out, _ = model.decoder(f3d)
        loss = LOSS(out, gt); loss.backward(); opt.step()
    if ep % EVAL_EVERY == 0 or ep == EPOCHS_SANITY - 1:
        model.decoder.train()                     # batch-stat BN (consistent with training) for the readout
        with torch.no_grad():
            ds = np.mean([per_bone_dice(model.decoder(f3d)[0], gt) for f3d, gt in samples], 0)
        hist.append((ep, float(loss), *ds))
        print(f"ep{ep:3d} loss {loss:.3f} | " + " ".join(f"{b} {ds[j]:.3f}" for j, b in enumerate(BONES)))
        if (ds >= TARGET_DICE).all():
            print(f"early stop @ep{ep}: all bones >= {TARGET_DICE}")
            break
print(f"trained in {time.time()-t0:.0f}s")

In [ ]:
# --- verdict + per-bone Dice curve + predicted-vs-GT montage ---
final = np.array(hist[-1][2:])
print("final per-bone Dice:", dict(zip(BONES, final.round(3))))

H = np.array([h[2:] for h in hist]); E = [h[0] for h in hist]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for j, b in enumerate(BONES):
    ax[0].plot(E, H[:, j], marker="o", label=b)
ax[0].axhline(0.9, ls="--", c="gray"); ax[0].set_ylim(0, 1.02)
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("per-bone Dice")
ax[0].set_title("Overfit smoke test (frozen front-end, decoder-only, 64^3)"); ax[0].legend(fontsize=8)

# montage: predicted vs GT mid-coronal slice per bone, for the first sanity knee
f3d0, gt0 = samples[0]
with torch.no_grad():
    pred = (torch.sigmoid(model.decoder(f3d0)[0][0].float()) > 0.5).cpu().numpy()   # (4,T,T,T)
gtn = gt0[0].cpu().numpy()
mid = TARGET_RES // 2
ax[1].axis("off"); ax[1].set_title(f"{SANITY_KEYS[0]}: GT (top) vs pred (bottom), mid-coronal", fontsize=9)
grid = np.zeros((2 * TARGET_RES, 4 * TARGET_RES))
for j in range(4):
    grid[:TARGET_RES, j*TARGET_RES:(j+1)*TARGET_RES] = gtn[j, :, mid, :]
    grid[TARGET_RES:, j*TARGET_RES:(j+1)*TARGET_RES] = pred[j, :, mid, :]
ax[1].imshow(grid, cmap="gray", origin="lower")
for j, b in enumerate(BONES):
    ax[1].text(j*TARGET_RES + 4, 2*TARGET_RES - 6, b, color="yellow", fontsize=8)
plt.tight_layout(); plt.savefig(ROOT / "reports" / "sanity_per_bone_dice.png", dpi=110); plt.show()

assert (final >= 0.85).all(), f"some bone underfit at {TARGET_RES}^3: {dict(zip(BONES, final.round(3)))}"
print("SANITY PASS: the encoder-decoder represents all 4 bones as independent labels.")